In [3]:
%load_ext autoreload
%autoreload 2

In [12]:
import numpy as np
import pandas as pd
from pyspark import SparkConf
from pyspark.sql import SparkSession
import pyspark.pandas as ps
import pyspark.sql.functions as F
import os

In [1]:
from hypex.matching import Matching
from hypex.ml.faiss import FaissNearestNeighbors
from hypex.transformers import TypeCaster
from hypex.dataset import Dataset, InfoRole, TreatmentRole, FeatureRole, TargetRole, ExperimentData, AdditionalMatchingRole
from hypex.utils import BackendsEnum
from hypex.experiments import Experiment, OnRoleExperiment
from hypex.comparators import MahalanobisDistance
from hypex.encoders.encoders import DummyEncoder
from hypex.comparators import TTest, Chi2Test
from hypex.comparators.distances import MahalanobisDistance
from hypex.operators import Bias, MatchingMetrics
from hypex.analyzers import MatchingAnalyzer
from hypex.utils import SparkSessionCalculator

In [4]:
# --- 1. Настройки окружения для macOS (Важно!) ---
# На macOS иногда возникают проблемы с форком процессов Java (Executor'ы не стартуют).
# Эта переменная часто решает проблему "Connection refused" или краши при запуске local-cluster
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

# Очистка старых сессий и переменных (как у вас было)
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("✅ Существующая сессия остановлена.")
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация Кластера ---
# Формат: local-cluster[число_воркеров, ядер_на_воркер, память_на_воркер_в_МБ]
# Мы просим 2 экзекутора, по 1 ядру, по 2 ГБ памяти каждый
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 4
MEMORY_PER_EXECUTOR_MB = 2048 

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"🚀 Запуск в режиме: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    # Память драйвера (остается у вас)
    .config("spark.driver.memory", "2g") 
    # Память экзекутора (должна соответствовать или быть меньше чем в master URL)
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "4")
    .config("spark.executor.instances", NUM_EXECUTORS)
    # Увеличиваем память под оверхед, чтобы избежать ошибок выделения памяти
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4") # Для тестов меньше дефолтных 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Проверка конфигурации ---
print(f"✅ Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Проверка количества экзекуторов (может занять пару секунд на старт)
import time
time.sleep(3) 
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 4. Тест на распределение (Пример) ---
# Чтобы убедиться, что задача ушла на экзекуторы, а не осталась на драйвере
def print_executor_info(iterator):
    import os
    # Получаем ID экзекутора из переменных окружения процесса
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Создаем датафрейм и применяем трансформацию
df = sp_s.range(0, 10, 1, 4) # 4 партиции
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

# Не забывайте останавливать сессию в конце скрипта, так как процессы тяжелые
# sp_s.stop() 

🚀 Запуск в режиме: local-cluster[2, 4, 2048]


26/07/10 11:56:19 WARN Utils: Your hostname, eric-Katana-17-B12UCR resolves to a loopback address: 127.0.1.1; using 10.240.76.27 instead (on interface wlo1)
26/07/10 11:56:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/10 11:56:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Сессия создана.
Driver Memory Config: 2g
Executor Memory Config: 2g


📊 Активных экзекуторов (проверка через RDD): 2



🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 717229
Executor ID: Driver/Local, PID: 717235
Executor ID: Driver/Local, PID: 717287
Executor ID: Driver/Local, PID: 717298


In [5]:
n = 5000  # увеличьте для теста IVF-индексов
df = pd.DataFrame({
    "treatment": np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
    "feat_num_1": np.random.normal(loc=10, scale=3, size=n),
    "feat_num_2": np.random.normal(loc=-2, scale=1.5, size=n),
    "feat_cat": np.random.choice(["A", "B", "C"], size=n),
    "target": np.random.normal(loc=100, scale=10, size=n)
})

In [10]:
nn = 200
index_df = pd.DataFrame(
    {
        '0': np.random.randint(0, 50, nn),
        '1': np.random.randint(0, 50, nn),
        '2': np.random.randint(0, 50, nn),
        '3': np.random.randint(0, 50, nn),
        '4': np.random.randint(0, 50, nn),
        'group': [0] * (nn//4) + [1] * (nn//4) + [2] * (nn//4) + [3] * (nn//4)
    }
)

# index_df = pd.concat([index_df, pd.DataFrame(data={
#     '0': np.nan,
#     '1': np.nan,
#     '2': np.nan,
#     '3': np.nan,
#     '4': np.nan,
#     'group': np.nan
# }, index=[0])]).reset_index(drop=True)
index_df

,0,1,2,3,4,group
0,10,49,49,30,6,0
1,21,3,34,0,1,0
2,46,30,48,0,4,0
3,13,36,9,5,23,0
4,29,6,41,21,28,0
...,...,...,...,...,...,...
195,4,36,9,23,3,3
196,5,20,19,21,9,3
197,10,13,38,48,7,3
198,38,46,24,47,19,3


In [11]:
session = (
            SparkSession.builder
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .config("spark.executor.memory", "4g")
            .config("spark.memory.fraction", "0.8") 
            .config("spark.memory.storageFraction", "0.3")
            # .config("spark.jars.packages", "ch.cern.sparkmeasure:spark-measure_2.12:0.23") 
            .getOrCreate()
          )

26/07/08 10:38:35 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [51]:
index_ds = Dataset(
    roles={
        'group': TargetRole()
    },
    # data=index_df
    data=session.createDataFrame(index_df),
    # data=sp_s.createDataFrame(index_df),
    session=session
    # session=sp_s
)

index_ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all d

,0,1,2,3,4,group
0,18,24,37,2,16,0
1,22,43,11,1,43,0
2,49,12,13,40,31,0
3,28,21,7,2,13,0
4,16,16,34,37,17,0
...,...,...,...,...,...,...
195,34,27,16,45,37,3
196,25,24,16,42,16,3
197,18,47,31,36,40,3
198,7,10,19,28,13,3


In [ ]:
"""
PANDAS case
"""
# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=df,
    backend=BackendsEnum.pandas,
)

pandas_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=2,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        # MatchingMetrics(
        #         grouping_role=TreatmentRole(),
        #         target_roles=[TargetRole()],
        #         metric="ate",
        #         n_neighbors=2,
        # ),
        # MatchingAnalyzer(),
        # OnRoleExperiment(
        #     executors=[
        #         TTest(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole(),
        #         ),
        #         Chi2Test(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole()
        #         )
        #     ],
        #     role=FeatureRole()
        # )
    ]
)
pandas_result = pandas_experiment.execute(ExperimentData(dataset))

/home/eric/HypEx/HypEx/hypex/dataset/backends/pandas_backend.py:1175: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
/home/eric/HypEx/HypEx/hypex/dataset/backends/pandas_backend.py:1175: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
/home/eric/HypEx/HypEx/hypex/dataset/backends/pandas_backend.py:1175: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
/home/eric/HypEx/HypEx/hypex/dataset/backends/pandas_backend.py:1175: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[gro

In [10]:
pandas_result.additional_fields.roles

{'MatchingMetrics┴┴': AdditionalTarget(<class 'float'>),
 'FaissNearestNeighbors┴┴┴0': AdditionalMatching(<class 'int'>),
 'DummyEncoder┴┴_feat_cat_B': AdditionalFeature(data_type=<class 'float'>),
 "Bias┴┴['target', 'target_matched']": AdditionalStatistic(<class 'float'>),
 'FaissNearestNeighbors┴┴┴1': AdditionalMatching(<class 'int'>),
 'DummyEncoder┴┴_feat_cat_C': AdditionalFeature(data_type=<class 'float'>)}

In [11]:
pandas_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error   P-value  CI Lower  CI Upper
 ATT    -0.170323        0.371679  0.646772 -0.898814  0.558169
 ATC    -0.448111        0.490087  0.360533 -1.408681  0.512459
 ATE    -0.337884        0.373918  0.366190 -1.070763  0.394994
 
 3 rows × 5 columns,
 'GroupTTest┴┴feat_num_1':                   p-value  statistic  pass
 (0,)┆feat_num_1  0.998892   0.001389   0.0
 (1,)┆feat_num_1  0.980956  -0.023872   0.0
 
 2 rows × 3 columns,
 'GroupTTest┴┴feat_num_2':                   p-value  statistic  pass
 (0,)┆feat_num_2  0.974840   0.031540   0.0
 (1,)┆feat_num_2  0.985358  -0.018353   0.0
 
 2 rows × 3 columns,
 'GroupTTest┴┴DummyEncoder||_feat_cat_B':                                  p-value  statistic  pass
 (0,)┆DummyEncoder┴┴_feat_cat_B  0.956230  -0.054887   0.0
 (1,)┆DummyEncoder┴┴_feat_cat_B  0.891579  -0.136315   0.0
 
 2 rows × 3 columns,
 'GroupTTest┴┴DummyEncoder||_feat_cat_C':                                  p-value  statistic  p

In [13]:
calculator = SparkSessionCalculator(
    data_size_bytes=10_000_000_000,  # 10 GB
    num_columns=8,
    num_categorical_columns=2,
    target_executor_cores=4,
    target_executor_memory_gb=4.0
)

# Вариант 1: Оптимизация существующего SparkConf
original_conf = (
    SparkConf()
    .setAppName("MyApp")
    .setMaster("yarn")
    .set("spark.executor.memory", "2g")  # Неоптимально
    .set("spark.executor.cores", "8")    # Слишком много ядер
)

optimized_conf = calculator.optimize_config(original_conf)


ЛОГ ОПТИМИЗАЦИИ КОНФИГА SPARK-СЕССИИ

📊 ИТОГО ИЗМЕНЕНИЙ: 13
   ➕ Добавлено: 10
   🔄 Изменено: 3

--------------------------------------------------------------------------------

🔄 [ИЗМЕНЕНО] 1. spark.executor.instances
   Было: 2
   Стало: 15
   Причина: Оптимальное количество executor'ов для параллельной обработки 63 партиций. Обеспечивает баланс между параллелизмом и накладными расходами.

🔄 [ИЗМЕНЕНО] 2. spark.executor.cores
   Было: 8
   Стало: 4
   Причина: Ограничение ядер на executor для предотвращения OOM при одновременной загрузке нескольких индексов FAISS в память. Рекомендуется ≤4 ядра.

➕ [ДОБАВЛЕНО] 3. spark.executor.memoryOverhead
   Стало: 384m
   Причина: Overhead памяти для JVM, сериализации и off-heap операций. 10% от executor memory, минимум 384 MB.

🔄 [ИЗМЕНЕНО] 4. spark.sql.shuffle.partitions
   Было: 4
   Стало: 63
   Причина: Количество партиций для shuffle-операций. Оптимизировано для размера данных 9.3 GB, обеспечивает баланс между параллелизмом и размером ин

In [11]:
# Например: 10_000_000 строк × 8 колонок × 8 байт = 640_000_000 байт (~0.6 ГБ)
data_size_bytes = 10_000_000 * 8 * 8  # если известно
calculator = SparkSessionCalculator(
    data_size_bytes=data_size_bytes,
    num_columns=8,
    num_categorical_columns=1
)

optimal_settings = calculator.calculate_optimal_settings()
current = calculator.check_current_settings(sp_s)
calculator.generate_recommendations(current, optimal_settings)
calculator.print_recommendations()


РЕКОМЕНДАЦИИ ПО НАСТРОЙКЕ SPARK-СЕССИИ

🟠 [HIGH] spark.sql.shuffle.partitions
   Текущее значение: 4
   Рекомендуемое значение: 10
   Причина: Оптимизируйте количество партиций для баланса между размером индекса и параллелизмом

🟡 [MEDIUM] spark.serializer
   Текущее значение: unknown
   Рекомендуемое значение: org.apache.spark.serializer.KryoSerializer
   Причина: Используйте KryoSerializer для более эффективной сериализации



In [ ]:
from hypex.utils.spark_config import SparkSessionCalculator, SparkSettings

# Вариант 1: Создание новой сессии с оптимальными настройками
calculator = SparkSessionCalculator(
    data_size_bytes=10_000_000_000,  # 10 GB
    num_columns=8,
    num_categorical_columns=2,
    target_executor_cores=4,
    target_executor_memory_gb=4.0
)

settings = calculator.calculate_optimal_settings()
spark = calculator.create_optimal_session(settings)

# Вариант 2: Проверка и оптимизация существующей сессии
spark = SparkSession.builder.getOrCreate()
calculator = SparkSessionCalculator(num_rows=1_000_000, num_columns=8)

current = calculator.check_current_settings(spark)
optimal = calculator.calculate_optimal_settings()
calculator.generate_recommendations(current, optimal)
calculator.print_recommendations()
calculator.apply_settings(spark, optimal)

In [9]:
"""
PYSPARK case
"""
# 3. Конвертация в Spark + ОБЯЗАТЕЛЬНАЯ колонка `index` (требование Faiss)
spark_df = sp_s.createDataFrame(df)

# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=spark_df,
    # data=session.createDataFrame(df),
    # data=df,
    backend=BackendsEnum.spark,
    session=sp_s,
    # session=session
)

spark_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=2,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        MatchingMetrics(
                grouping_role=TreatmentRole(),
                target_roles=[TargetRole()],
                metric="ate",
                n_neighbors=2,
        ),
        MatchingAnalyzer(),
        # OnRoleExperiment(
        #     executors=[
        #         TTest(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole(),
        #         ),
        #         Chi2Test(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole()
        #         )
        #     ],
        #     role=FeatureRole()
        # )
    ]
)
spark_result = spark_experiment.execute(
    ExperimentData(dataset)
)

2026-07-09 17:19:39 | INFO     | hypex.experiment | ============================================================
2026-07-09 17:19:39 | INFO     | hypex.experiment | Spark Session Info:
2026-07-09 17:19:39 | INFO     | hypex.experiment |   Master: local-cluster[2, 4, 2048]
2026-07-09 17:19:39 | INFO     | hypex.experiment |   App name: LocalClusterTest
2026-07-09 17:19:39 | INFO     | hypex.experiment |   Driver memory: 2g
2026-07-09 17:19:39 | INFO     | hypex.experiment |   Executor memory: 2g
2026-07-09 17:19:39 | INFO     | hypex.experiment |   Executor cores: 4
2026-07-09 17:19:39 | INFO     | hypex.experiment |   Executor instances: 8
2026-07-09 17:19:39 | INFO     | hypex.experiment |   Spark version: 3.5.1
2026-07-09 17:19:39 | INFO     | hypex.experiment | ============================================================
2026-07-09 17:19:39 | INFO     | hypex.experiment | ▶ Process started: DummyEncoder [spark]
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:10

In [9]:
a = spark_result.field_search(InfoRole()) or None
a is None

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


True

In [12]:
spark_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error   P-value  CI Lower  CI Upper
 ATT    -0.213129        0.980223  0.827874 -2.134366  1.708108
 ATC     0.000490        0.824139  0.999525 -1.614821  1.615802
 ATE    -0.098048        0.773503  0.899131 -1.614114  1.418018
 
 3 rows × 5 columns}

In [7]:
spark_result.ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,Bias┴┴target_bias,Bias┴┴target_matched_target
2,1,14.208973,-1.194687,B,93.144985,1.0,0.0,2769,2005,-1.850296,94.99792
4,0,9.021018,-1.372699,C,104.130018,0.0,1.0,1205,2431,15.591174,88.517108
5,0,9.043075,-0.657164,C,89.267936,0.0,1.0,496,4435,-22.602262,111.901704
8,1,5.938445,-1.210908,A,107.483695,0.0,0.0,4927,545,10.747533,96.720872
12,0,11.028771,-1.540009,C,84.923102,0.0,1.0,2297,4806,-19.426577,104.376759
...,...,...,...,...,...,...,...,...,...,...,...
4995,0,9.562625,-4.569026,C,82.533743,0.0,1.0,577,4044,-21.166491,103.729759
4996,1,11.728486,-1.896468,B,106.871564,1.0,0.0,672,3066,11.12951,95.726223
4997,1,10.200868,-2.082435,B,92.33909,1.0,0.0,3122,982,-0.148631,92.48793
4998,0,4.210678,-0.564392,A,95.937813,0.0,0.0,1896,2221,-1.605121,97.545157


In [19]:
spark_result.ds.unpersist()

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,Bias┴┴target_bias,Bias┴┴target_target_matched,MatchingMetrics┴┴
2,0,12.045232,-3.791779,B,96.912328,1.0,0.0,3451,4010,-1.752359,98.667182,98.667182
4,0,7.5948,-3.506998,C,97.817065,0.0,1.0,662,639,-0.725902,98.54402,98.54402
5,0,11.261949,-2.80301,C,83.018658,0.0,1.0,3132,3036,-17.915917,100.960448,100.960448
8,1,11.935777,-1.693673,C,107.441409,0.0,1.0,4609,1616,13.27009,94.152028,94.152028
12,0,16.536284,-4.516249,B,96.54423,1.0,0.0,2819,3982,-6.57412,103.128048,103.128048
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,1,13.395537,-2.90735,B,92.776833,1.0,0.0,2244,3676,-5.429742,98.214405,98.214405
4996,0,12.690551,1.248961,B,105.992565,1.0,0.0,3858,2290,7.362271,98.619517,98.619517
4997,1,14.991156,-0.602245,C,99.631197,0.0,1.0,2338,183,10.378529,89.237694,89.237694
4998,1,8.153213,-0.34911,B,111.509293,1.0,0.0,3763,4132,10.817036,100.676567,100.676567


In [12]:
spark_result.variables["MahalanobisDistance┴┴['feat_num_1', 'feat_num_2', 'DummyEncoder||_feat_cat_B', 'DummyEncoder||_feat_cat_C']"]["['feat_num_1', 'feat_num_2', 'DummyEncoder┴┴_feat_cat_B', 'DummyEncoder┴┴_feat_cat_C']"]

,feat_num_1,feat_num_2,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C
feat_num_1,0.329423,0.005064,-0.002166,-0.001165
feat_num_2,0.000000,0.674589,0.004112,0.004982
DummyEncoder┴┴_feat_cat_B,0.000000,0.000000,2.103345,1.198875
DummyEncoder┴┴_feat_cat_C,0.000000,0.000000,0.000000,2.456418


In [15]:
spark_result.field_search(AdditionalStatisticRole())

[]

In [7]:
spark_result.ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,"Bias┴┴['target', 'target_matched']",MatchingMetrics┴┴
2507,1,9.344322,0.061681,C,122.704119,0.0,1.0,2593,3491,0.001844,87.343549
2509,1,15.051634,-1.646648,C,103.488801,0.0,1.0,1057,2774,0.007622,109.718559
2513,0,6.778776,-3.909684,B,99.504827,1.0,0.0,4593,1582,0.001345,104.010242
2529,1,10.611894,-1.060253,B,101.451064,1.0,0.0,1125,4441,-0.006783,92.709322
2532,0,13.309939,-3.323301,A,98.405159,0.0,0.0,4522,3631,0.007523,87.343303
...,...,...,...,...,...,...,...,...,...,...,...
4995,0,4.215119,-1.728702,A,116.791128,0.0,0.0,1437,3900,0.022995,93.282247
4996,0,9.728035,-1.025706,C,104.852101,0.0,1.0,597,463,0.022611,100.832959
4997,1,5.080752,-2.045856,C,94.247351,0.0,1.0,3223,2161,-0.027136,88.786346
4998,0,10.767351,-0.634082,A,88.233454,0.0,0.0,4690,3723,0.006813,110.698601


In [ ]:
# for label, ds in result.variables['Bias┴┴[\'target\', \'target_matched\']'].items():
#     ds.to_small_dataset().data.to_csv(f"{label}.csv")

In [6]:
spark_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error  P-value  CI Lower  CI Upper
 ATT    -0.386312             0.0      0.0 -0.386312 -0.386312
 ATC     0.223290             0.0      0.0  0.223290  0.223290
 ATE    -0.024031             0.0      0.0 -0.024031 -0.024031
 
 3 rows × 5 columns}

In [8]:
sp_s.stop()

In [ ]:
r = pd.read_csv('result_[5].csv', names=['index', 'value'])
r.dropna(subset=['index']).fillna(0)

In [ ]:

# 5. Настройка Matching
matching = Matching(
    distance="mahalanobis",
    # metric="ate",
    bias_estimation=False,      # отключаем для упрощения дебага
    quality_tests=["t-test"],   # только t-test для скорости
    faiss_mode="base",          # "base" → IndexFlatL2 (без IVF), проще отлаживать
    n_neighbors=1,
    encode_categories=True      # DummyEncoder включится автоматически
)

# 6. Запуск
print("🚀 Запуск пайплайна Matching...")
result_data = matching.execute(dataset)

print("✅ Выполнено успешно!")
print(f"📊 Additional Fields: {result_data.additional_fields.columns}")
print(f"📦 Groups Keys: {list(result_data.groups.keys())}")

In [ ]:
sp_s.stop()